<a href="https://colab.research.google.com/github/fredstridsh/gestamp_ml/blob/main/data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing for Machine Learning

In machine learning, algorithms learn from data, but real-world data is, as we all not, notoriously messy. It contains missing values, vast differences in numeric scales, and text variables that algorithms cannot mathematically process.

**The Golden Rule:** The quality of your preprocessing dictates the quality of your model ("garbage in, garbage out").

In [ ]:
# Pandas and numpy are used for data structures and mathematics
# matplotlib.pyplot for plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the real world, Titanic dataset directly from a public URL
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
data = pd.read_csv(url)

print("Dataset loaded successfully! Here are the first 5 rows:")
display(data.head())

## 1. Handling Missing Data
Real-world data is rarely perfect. It often contains missing values (represented as Not-a-Number (`NaN`), in Python). We can handle missing data in a few ways:

* **Dropping:** Remove rows or columns with missing data.
    * *Benefit:* Easy to do, guarantees no fabricated data.
    * *Drawback:* Severe loss of potentially valuable information if many rows are dropped. May lead to incorrect conclusions
* **Imputation:** Fill in missing values with a statistical measure (mean, median, mode) or a constant.
    * *Benefit:* Keeps your dataset size intact.
    * *Drawback:* Can artificially distort the feature's variance or relationships with other variables.

In [ ]:
from sklearn.impute import SimpleImputer

print(f"Missing 'Age' values before: {data['Age'].isna().sum()}")

# Try different parameters to see how the blanks are filled.
# Strategies available: mean, median, and most_frequent
IMPUTATION_STRATEGY = 'mean'

# Initialize the imputer and apply it
imputer = SimpleImputer(strategy=IMPUTATION_STRATEGY)
data['Age_Imputed'] = imputer.fit_transform(data[['Age']])

print(f"Missing 'Age_Imputed' values after: {data['Age_Imputed'].isna().sum()}")
print("\nNotice how the NaN values in the left column are replaced on the right:")
display(data[['Age', 'Age_Imputed']].head(10))

## 2. Feature Scaling
Machine learning models (like Neural Networks) often perform poorly if numerical features are on entirely different scales (e.g., Passenger Age ranges from 0 to 80, but Ticket Fare ranges from 0 to 500). We there often scale the features before testing any models.

* **Standardization (`StandardScaler`):** Centers data around a mean of 0 and a standard deviation of 1.
    * *Benefit:* Highly robust to outliers. Preserves the underlying distribution shape.
* **Normalization (`MinMaxScaler`):** Scales data to a fixed, strict range, usually exactly 0.0 to 1.0.
    * *Benefit:* Suitable for algorithms that require strict boundaries (like image processing).

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# test both StandardScaler and MinMaxScaler and observe how the X-axis changes on the right-hand graph:
scaler = StandardScaler()
# scaler = MinMaxScaler()

# Apply scaling to the 'Fare' column
data['Fare_Scaled'] = scaler.fit_transform(data[['Fare']])

# Plotting the difference
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(data['Fare'], bins=30, color='blue', alpha=0.7)
ax[0].set_title('Original Fare (Wide Scale: 0 to 500)')

ax[1].hist(data['Fare_Scaled'], bins=30, color='green', alpha=0.7)
ax[1].set_title(f'Scaled Fare using {scaler.__class__.__name__}')
plt.show()

## 3. Encoding Categorical Variables
Machine learning models are similar to mathematical equations. To work,
they require not text but numbers. We must convert categorical text data (like 'Male'/'Female' or 'City A'/'City B') into numerical formats.

* **Label/Ordinal Encoding:** Assigns an integer to each category (e.g., 0, 1, 2).
    * *Drawback:* Models might falsely assume a mathematical hierarchy (e.g., assumes 2 is "greater" than 0) where none exists. Best used only for inherently ordered data (e.g., Low, Medium, High).
* **One-Hot Encoding:** Creates a new binary column (0 or 1) for every unique category.
    * *Benefit:* Prevents the model from assuming any numerical order.
    * *Drawback:* Can significantly bloat the size of your dataset if a column has hundreds of unique text values.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# We will encode the 'Embarked' column (Port of Embarkation: C, Q, or S)
# C = Cherbourg (France)
# Q = Queenstown (Ireland)
# S = Southampton (UK)


# Try DROP_FIRST True or False
# True drops one column to prevent multicollinearity
DROP_FIRST = False

drop_param = 'first' if DROP_FIRST else None
encoder = OneHotEncoder(sparse_output=False, drop=drop_param)

# We drop NaNs just for this quick demonstration step
clean_ports = data[['Embarked']].dropna()
encoded_ports = encoder.fit_transform(clean_ports)

# Convert back to a readable DataFrame to show you what happened under the hood
encoded_df = pd.DataFrame(encoded_ports, columns=encoder.get_feature_names_out(['Embarked']))

print("Original Text Column (first 3 rows):")
display(clean_ports.head(3))

print("\nOne-Hot Encoded Columns:")
display(encoded_df.head(3))

## 4. Putting it Together: The Pipeline
In real-world problems, you don't apply these steps one by one manually. Applying them individually to training data, validation data, and incoming production data is tedious and creates massive room for bugs (like applying the wrong scaler to the wrong column, forgetting steps, etc).

A **Pipeline** bundles all preprocessing steps together into a single object. You configure it once, and it consistently and safely processes any raw data you feed into it.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Separate features by the type of processing they need
numerical_features = ['Age', 'Fare']
categorical_features = ['Embarked', 'Sex']

# 1. Define the recipe for numerical columns
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Define the recipe for categorical columns
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Combine both recipes using a ColumnTransformer
master_preprocessor = ColumnTransformer([
    ('num_processing', num_pipeline, numerical_features),
    ('cat_processing', cat_pipeline, categorical_features)
])

# Fetch the raw, messy data
raw_features = data[numerical_features + categorical_features]

# Execute the entire pipeline in one line of code!
processed_data = master_preprocessor.fit_transform(raw_features)

print("Shape of original data:", raw_features.shape)
print("Shape of fully processed data:", processed_data.shape)
print("\nFirst row of fully preprocessed data (Numbers only, scaled, and ready for an ML model!):")
print(np.round(processed_data[0], 3))